# How to batch-process data

There may be cases where you would like to generate data for several sets of filters on the same dataset. This how-to guide will show you how this can be done in MNeuEventLib.

This guide assumes you are familiar with filtering in MNeuEventLib. The [filtering how-to guide](filtering.ipynb) can teach you how to do this.

## Creating multiple filter objects

The most flexible way to batch-process data involves creating `Filters` objects independent of the data, and swapping them in and out as desired with the `Data` object's `set_filters` method. We can import the Filters object, which represents a set of filters. It has an interface similar to the filtering interface in `Data`.

In [ ]:
from MNeuEventLib import Filters

help(Filters)

Now, as an example, say we want to calculate histograms for multiple time slices of a dataset. We load in our data:

In [ ]:
from MNeuEventLib import Data

data = Data("../_static/HIFI00195790.nxs", n_spec=64)

We can get more information about the dataset by accessing `Data`'s `dataset` field. For example, if we are doing time slicing, we would like to know information about the time frames of our dataset. The start time of each frame (in nanoseconds) is available through the method `get_frame_times`. We will divide these times by $10^9$ to get the times in seconds.

In [ ]:
frame_start_times = data.dataset.get_frame_times() / 1e9
print(frame_start_times)

Now we want to split these 88 frames into 4 segments. We can create filters for all four segments with a `for` loop:

In [ ]:
n_segments = 4  # change this if you want more or fewer segments!

# calculate our segment size and set up a list to hold our filters...
n_frames = len(frame_start_times)
segment_size = n_frames // n_segments
filter_sets = []

# create each segment
for segment in range(0, n_segments):
    # this figures out the time range for each segment,
    # e.g. in 4 segments:
    # for the first segment we want frame 0 through to 21,
    # for the second we want 22 through to 43...
    start_time = frame_start_times[segment_size * segment]
    end_time = frame_start_times[segment_size * (segment + 1) - 1]

    # now we create a filter for each segment! 
    # note the default filter type is `include`,
    # so this filter means 'only include data in this time range'
    f = Filters()
    f.add_time_filter("filter", start_time, end_time)
    filter_sets.append(f)

and then again we can use a `for` loop to run the calculation for each filter set in turn. We add the `enumerate` function to our loop so we can label each output file separately.

In [ ]:
# for each segment, set the data object's filters to that segment,
# and then save to a file
for k, filter_set in enumerate(filter_sets):
    data.set_filters(filter_set)
    data.calculate()
    print(f"Calculated segment {k+1} with {data.results.n_events()} events")
    data.save(f"output_{k+1}.nxs")